# Pharmasense — ST-GNN Training
**Spatio-Temporal Graph Neural Network for communicable disease outbreak prediction**

## What this model covers
- **Diseases:** `dengue`, `malaria`, `cholera` — communicable, vector/waterborne, spread geographically via district adjacency
- **NOT** `acute_diarrhoeal_disease` → that is endemic/seasonal, handled by TFT (notebook 02)
- **Architecture:** GraphSAGE (2 hops spatial) + GRU (24-week temporal) → regression on normalized future case counts
- **Output:** `stgnn_latest.pt` saved to Google Drive + auto-downloaded

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. **Add Colab Secret:** click 🔑 in the left sidebar → add `PHARMASENSE_DB_PASSWORD` = your DB password
3. **Runtime → Run All**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/01_train_stgnn.ipynb)

In [ ]:
# ── 0a. Check GPU ─────────────────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU: {result.stdout.strip()}')
else:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 0b. Install dependencies ──────────────────────────────────────────────────
# torch 2.2 + CUDA 11.8 wheels (Colab T4 compatible)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.2.0', 'torchvision', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)

# PyTorch Geometric + sparse ops (must match torch version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric',
    'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
    '-f', 'https://data.pyg.org/whl/torch-2.2.0+cu118.html'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'psycopg2-binary', 'pandas', 'numpy', 'scikit-learn', 'tqdm', 'matplotlib'], check=True)

print('All dependencies installed.')

In [ ]:
# ── 1. Config ─────────────────────────────────────────────────────────────────
# Credentials via Colab Secrets (never hardcoded)
# Add these in the 🔑 sidebar: PHARMASENSE_DB_PASSWORD
from google.colab import userdata

DB_CONFIG = dict(
    host='aws-1-ap-south-1.pooler.supabase.com', port=6543,
    dbname='postgres', user='postgres.xwtvujwdybvjbaucshbt',
    password=userdata.get('PHARMASENSE_DB_PASSWORD'), sslmode='require',
)

# ── Disease scope (ST-GNN = communicable, spatially spreading) ─────────────────
# ADD (acute_diarrhoeal_disease) is endemic/seasonal — goes to TFT notebook 02
STGNN_DISEASES = ['dengue', 'malaria', 'cholera']

# ── Model hyperparameters ─────────────────────────────────────────────────────
LOOKBACK_WEEKS  = 24   # 6 months — captures annual disease seasonality cycles
HORIZON_WEEKS   = 2    # predict 2 weeks (14 days) ahead — matches product promise
HIDDEN_DIM      = 64   # graph hidden dimension (empirically fine for N=641)
NUM_SAGE_LAYERS = 2    # 2-hop neighborhood = 2 weeks of potential spread propagation
GRU_LAYERS      = 2    # 2-layer GRU for 24-step sequence
DROPOUT         = 0.2  # reduced from 0.3 — dataset is sparse, less regularisation needed
LEARNING_RATE   = 1e-3
WEIGHT_DECAY    = 1e-4
EPOCHS          = 150  # raised ceiling — early stop handles actual termination
EARLY_STOP_PAT  = 20   # patience: stop if val loss doesn't improve for 20 epochs
TRAIN_SPLIT     = 0.75
VAL_SPLIT       = 0.15  # test = remaining 0.10

# Output
DRIVE_OUTPUT    = '/content/drive/MyDrive/pharmasense_models/'

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Training diseases: {STGNN_DISEASES}')
print(f'Lookback: {LOOKBACK_WEEKS} weeks | Horizon: {HORIZON_WEEKS} weeks')

In [ ]:
# ── 2. Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive ready. Artifacts → {DRIVE_OUTPUT}')

In [ ]:
# ── 3. Load data from Supabase ────────────────────────────────────────────────
# Two bulk SELECT queries — total data ~250KB. Latency: 2-3 seconds. Not an issue.
import psycopg2
import pandas as pd
import numpy as np

conn = psycopg2.connect(**DB_CONFIG)

districts_df = pd.read_sql("""
    SELECT id, name, state, centroid_lat, centroid_lon, adjacency_ids
    FROM districts ORDER BY id;
""", conn)

# Only fetch ST-GNN diseases — dengue, malaria, cholera
disease_filter = "', '".join(STGNN_DISEASES)
epid_df = pd.read_sql(f"""
    SELECT district_id, disease, week_start_date, cases
    FROM epid_weekly
    WHERE disease IN ('{disease_filter}')
    ORDER BY week_start_date, district_id, disease;
""", conn)
conn.close()

epid_df['week_start_date'] = pd.to_datetime(epid_df['week_start_date'])

print(f'Districts: {len(districts_df)}')
print(f'Epid records (communicable only): {len(epid_df)}')
print(f'Date range: {epid_df["week_start_date"].min().date()} -> {epid_df["week_start_date"].max().date()}')
print()
print(epid_df.groupby('disease')['cases'].agg(['count','sum']).rename(columns={'count':'weeks','sum':'total_cases'}))

In [ ]:
# ── 4. Build graph + spatio-temporal feature tensors ─────────────────────────
from sklearn.preprocessing import StandardScaler

# District index (consistent ordering)
dist_ids  = list(districts_df['id'])
id_to_idx = {d: i for i, d in enumerate(dist_ids)}
N         = len(dist_ids)   # 641
D_in      = len(STGNN_DISEASES)  # 3

# Build edge_index from adjacency_ids
src_list, dst_list = [], []
for _, row in districts_df.iterrows():
    i = id_to_idx[row['id']]
    for nbr in (row['adjacency_ids'] or []):
        j = id_to_idx.get(nbr)
        if j is not None:
            src_list.append(i)
            dst_list.append(j)

edge_index = torch.tensor([src_list, dst_list], dtype=torch.long, device=DEVICE)
print(f'Graph: N={N} nodes, E={edge_index.shape[1]} directed edges, avg_degree={edge_index.shape[1]/N:.1f}')

# Time axis: union of all weeks in data
all_weeks   = sorted(epid_df['week_start_date'].unique())
T           = len(all_weeks)
week_to_idx = {w: i for i, w in enumerate(all_weeks)}

# Feature tensor: [T, N, D_in]
X = np.zeros((T, N, D_in), dtype=np.float32)
for row in epid_df.itertuples():
    t = week_to_idx.get(row.week_start_date)
    n = id_to_idx.get(row.district_id)
    d = STGNN_DISEASES.index(row.disease) if row.disease in STGNN_DISEASES else None
    if t is not None and n is not None and d is not None:
        X[t, n, d] += row.cases

# log1p transform to handle extreme outbreak values, then z-score per disease
X = np.log1p(X)
scalers = []
for d in range(D_in):
    sc = StandardScaler()
    flat = X[:, :, d].reshape(-1, 1)
    X[:, :, d] = sc.fit_transform(flat).reshape(T, N)
    scalers.append(sc)

print(f'Feature tensor: {X.shape}  (T weeks × N districts × {D_in} diseases)')
print(f'Non-zero timestep-district-disease entries: {(X != 0).sum():,} / {X.size:,} '
      f'({100*(X!=0).sum()/X.size:.1f}% fill rate)')

# Target: predict state HORIZON_WEEKS ahead
Y = np.roll(X, -HORIZON_WEEKS, axis=0)  # [T, N, D] — future state
Y[-HORIZON_WEEKS:] = 0  # zero out invalid future steps

In [ ]:
# ── 5. ST-GNN model definition ────────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class STGNN(nn.Module):
    """
    Spatio-Temporal GNN — GraphSAGE + GRU.

    Design rationale:
    - GraphSAGE chosen over GCN: inductive (can generalise to unseen districts),
      no need to recompute eigenvectors when graph changes, better for heterogeneous
      node features (districts differ massively in population density).
    - 2 SAGE layers = information from 2-hop neighborhood = district + its neighbors'
      neighbors. For vector-borne disease with ~1 week travel time, this is correct.
    - GRU over LSTM: less parameters, similar performance for sequences <50 steps,
      faster to train on sparse epidemiological data.

    Input:  x_seq [T_lookback, N, D_in]
    Output: pred  [N, D_out]  — predicted normalized future case counts
    """
    def __init__(self, in_ch, hidden, out_ch, n_sage=2, n_gru=2, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        # Spatial: GraphSAGE message passing
        self.sage = nn.ModuleList([SAGEConv(in_ch if i == 0 else hidden, hidden)
                                   for i in range(n_sage)])
        self.sage_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_sage)])

        # Temporal: GRU over lookback window
        self.gru = nn.GRU(hidden, hidden, num_layers=n_gru,
                          batch_first=True,
                          dropout=dropout if n_gru > 1 else 0.0)

        # Prediction head
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, out_ch)
        )

    def forward(self, x_seq, edge_index):
        T, N, _ = x_seq.shape
        spatial_out = []
        for t in range(T):
            h = x_seq[t]  # [N, D_in]
            for layer, norm in zip(self.sage, self.sage_norms):
                h = norm(F.gelu(layer(h, edge_index)))
                h = F.dropout(h, p=self.dropout, training=self.training)
            spatial_out.append(h)  # [N, hidden]

        seq = torch.stack(spatial_out, dim=1)   # [N, T, hidden]
        gru_out, _ = self.gru(seq)              # [N, T, hidden]
        return self.head(gru_out[:, -1, :])     # [N, out_ch]


model = STGNN(D_in, HIDDEN_DIM, D_in,
               n_sage=NUM_SAGE_LAYERS, n_gru=GRU_LAYERS, dropout=DROPOUT).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ST-GNN  |  params: {total_params:,}  |  hidden: {HIDDEN_DIM}  |  SAGE layers: {NUM_SAGE_LAYERS}')
print(model)

In [ ]:
# ── 6. Train / Val / Test split + training loop ───────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Sliding window samples: X[t:t+LOOKBACK], Y[t+LOOKBACK]
samples = [
    (X[t:t + LOOKBACK_WEEKS].copy(), Y[t + LOOKBACK_WEEKS - 1].copy())
    for t in range(T - LOOKBACK_WEEKS - HORIZON_WEEKS)
]

n_train = int(len(samples) * TRAIN_SPLIT)
n_val   = int(len(samples) * VAL_SPLIT)
train_s = samples[:n_train]
val_s   = samples[n_train:n_train + n_val]
test_s  = samples[n_train + n_val:]
print(f'Samples — Train: {len(train_s)}, Val: {len(val_s)}, Test: {len(test_s)}')

optimizer  = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion  = nn.HuberLoss(delta=1.0)  # Huber loss: robust to outlier outbreaks
ei_dev     = edge_index.to(DEVICE)

train_losses, val_losses = [], []
best_val, best_state, patience_ctr = float('inf'), None, 0

for epoch in tqdm(range(EPOCHS), desc='Training STGNN'):
    model.train()
    np.random.shuffle(train_s)
    ep_loss = sum(
        (lambda x, y: (optimizer.zero_grad(),
                       (loss := criterion(model(torch.tensor(x, device=DEVICE), ei_dev),
                                          torch.tensor(y, device=DEVICE))),
                       loss.backward(),
                       nn.utils.clip_grad_norm_(model.parameters(), 1.0),
                       optimizer.step(), loss.item())[-1])(*s)
        for s in train_s
    ) / len(train_s)
    train_losses.append(ep_loss)

    model.eval()
    with torch.no_grad():
        vl = sum(criterion(model(torch.tensor(x, device=DEVICE), ei_dev),
                           torch.tensor(y, device=DEVICE)).item()
                 for x, y in val_s) / len(val_s)
    val_losses.append(vl)
    scheduler.step()

    if vl < best_val:
        best_val   = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= EARLY_STOP_PAT:
            print(f'Early stop at epoch {epoch+1}  |  best val: {best_val:.5f}')
            break

    if (epoch + 1) % 20 == 0:
        print(f'E{epoch+1:3d}  train={ep_loss:.5f}  val={vl:.5f}  best={best_val:.5f}')

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train (Huber)'), plt.plot(val_losses, label='Val (Huber)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('ST-GNN Loss'); plt.legend()
plt.tight_layout(); plt.savefig(DRIVE_OUTPUT + 'stgnn_loss.png'); plt.show()

In [ ]:
# ── 7. Test set evaluation ─────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, r2_score

model.load_state_dict(best_state)
model.eval()

all_preds, all_tgts = [], []
with torch.no_grad():
    for x_np, y_np in test_s:
        pred = model(torch.tensor(x_np, device=DEVICE), ei_dev).cpu().numpy()
        all_preds.append(pred); all_tgts.append(y_np)

P = np.vstack(all_preds); T_arr = np.vstack(all_tgts)
print('Test Metrics (normalized log-case space):')
print(f'  Overall MAE:  {mean_absolute_error(T_arr, P):.4f}')
print(f'  Overall R²:   {r2_score(T_arr.ravel(), P.ravel()):.4f}')
print()
for i, dis in enumerate(STGNN_DISEASES):
    mae = mean_absolute_error(T_arr[:, :, i].ravel(), P[:, :, i].ravel())
    r2  = r2_score(T_arr[:, :, i].ravel(), P[:, :, i].ravel())
    print(f'  {dis:10s}  MAE={mae:.4f}  R²={r2:.4f}')

# Risk score conversion (for inspection — not part of training)
# In production: inverse_transform(pred) -> log-cases -> cases_per_100k -> risk bucket
print('\nNote: production inference inverts log-normalisation to get actual case estimates')

In [ ]:
# ── 8. Save artifact ──────────────────────────────────────────────────────────
import json
from datetime import datetime

artifact = {
    'model_state_dict': best_state,
    'model_config': {
        'in_channels':      D_in,
        'hidden_channels':  HIDDEN_DIM,
        'out_channels':     D_in,
        'num_sage_layers':  NUM_SAGE_LAYERS,
        'num_gru_layers':   GRU_LAYERS,
        'dropout':          DROPOUT,
    },
    'training_config': {
        'diseases':         STGNN_DISEASES,
        'lookback_weeks':   LOOKBACK_WEEKS,
        'horizon_weeks':    HORIZON_WEEKS,
    },
    'graph': {
        'n_districts':  N,
        'id_to_idx':    id_to_idx,
        'n_edges':      edge_index.shape[1],
    },
    'normalisation': {
        'scalers':       scalers,
        'diseases_order': STGNN_DISEASES,
        'transform':     'log1p then StandardScaler per disease channel',
    },
    'eval': {
        'best_val_loss': best_val,
        'trained_at':    datetime.now().isoformat(),
        'n_train':       len(train_s),
        'n_val':         len(val_s),
        'n_test':        len(test_s),
    }
}

save_path = DRIVE_OUTPUT + 'stgnn_latest.pt'
torch.save(artifact, save_path)
print(f'Artifact saved: {save_path}')
print('Copy to:  ml/artifacts/stgnn_latest.pt  in your local repo')

from google.colab import files
files.download(save_path)
print('Download started.')